# RANST News Detection - Streaming Pipeline (Colab GPU)

Volledig geautomatiseerd: haalt laatste Ranst gemeenteraad video op, transcribeert, detecteert nieuws, downloadt resultaten.

Progress updates elke 5 minuten met percentage en geschatte resterende tijd.

## 1. SETUP

In [ ]:
import torch
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: Geen GPU! Ga naar Runtime > Change runtime type > T4 GPU')

In [ ]:
!pip install -q "numpy<2.0" "scipy<1.14" openai-whisper pyannote.audio yt-dlp tqdm

In [ ]:
!apt-get update -qq && apt-get install -y -qq ffmpeg 2>&1 | tail -1

## 2. CONFIG & IMPORTS

In [ ]:
import os
import json
import subprocess
import time as _time
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
import whisper
import torch
from tqdm import tqdm

try:
    from pyannote.audio import Pipeline
    DIARIZATION_AVAILABLE = True
except Exception as e:
    print(f'pyannote niet beschikbaar ({e}), diarization wordt overgeslagen')
    DIARIZATION_AVAILABLE = False

TEMP_DIR = '/content/temp'
OUTPUT_DIR = '/content/output'
os.makedirs(TEMP_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
FP16 = DEVICE == 'cuda'
WHISPER_MODEL = 'base'
NEWS_THRESHOLD = 0.65

# HuggingFace token (optioneel, voor sprekerherkenning)
HF_TOKEN = ""  # @param {type:"string"}
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    print('HF_TOKEN ingesteld')
else:
    print('Geen HF_TOKEN - diarization wordt overgeslagen')

print(f'Device: {DEVICE} | FP16: {FP16}')
print(f'Whisper model: {WHISPER_MODEL}')
print(f'Diarization: {"Beschikbaar" if DIARIZATION_AVAILABLE else "Niet beschikbaar"}')

## 3. CORE FUNCTIONS

In [ ]:
_whisper_model = None
_diarizer_pipeline = None

def get_whisper_model():
    global _whisper_model
    if _whisper_model is None:
        _whisper_model = whisper.load_model(WHISPER_MODEL, device=DEVICE)
    return _whisper_model

def get_diarizer():
    global _diarizer_pipeline
    if not DIARIZATION_AVAILABLE:
        return None
    if _diarizer_pipeline is None:
        hf_token = os.getenv('HF_TOKEN', '')
        if not hf_token:
            return None
        _diarizer_pipeline = Pipeline.from_pretrained(
            'pyannote/speaker-diarization-3.1',
            use_auth_token=hf_token
        ).to(DEVICE)
    return _diarizer_pipeline

def transcribe_chunk(audio_path):
    model = get_whisper_model()
    result = model.transcribe(
        audio_path, language='nl', task='transcribe',
        verbose=False, word_timestamps=False,
        fp16=FP16, temperature=0.3, beam_size=5
    )
    return result['segments']

def diarize_chunk(audio_path):
    pipeline = get_diarizer()
    if pipeline is None:
        return []
    diarization = pipeline(audio_path)
    turns = []
    for turn, _, speaker in diarization.itertracks(yield_label=True):
        turns.append({'speaker': speaker, 'start': round(turn.start, 2), 'end': round(turn.end, 2)})
    return turns

def match_speaker_to_segment(segment, turns):
    if not turns:
        return 'Speaker'
    seg_start = segment['start']
    for turn in turns:
        if turn['start'] <= seg_start < turn['end']:
            return turn['speaker']
    return 'Unknown'

def detect_news(text):
    news_keywords = [
        'budget', 'besluit', 'wet', 'regel', 'voorstel', 'motie',
        'financieel', 'miljoen', 'bouw', 'project', 'belangrijk',
        'subsidie', 'belasting', 'stemming', 'amendement', 'bezwaar',
        'commissie', 'vergunning', 'bestemmingsplan', 'meerderheid'
    ]
    text_lower = text.lower()
    matches = [kw for kw in news_keywords if kw in text_lower]
    score = min(len(matches) * 0.15, 1.0)
    reason = f'Keywords: {chr(44).join(matches[:5])}' if matches else 'Geen keywords'
    return {
        'is_newsworthy': score > NEWS_THRESHOLD,
        'score': round(score, 2),
        'category': 'politiek',
        'reason': reason
    }

print('Functions loaded')

## 4. DOWNLOAD VIDEO

In [ ]:
import yt_dlp

YOUTUBE_CHANNEL = 'https://www.youtube.com/@gemeenteranst1107/videos'

print('Zoeken naar laatste video van gemeente Ranst...')

ydl_opts = {
    'quiet': True, 'no_warnings': True,
    'socket_timeout': 30, 'playlist_items': '1-5',
}

video_entry = None
with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(YOUTUBE_CHANNEL, download=False)
    if info and 'entries' in info:
        for entry in info['entries']:
            if entry and 'id' in entry:
                video_entry = entry
                break

if not video_entry:
    raise RuntimeError('Geen video gevonden op het Ranst kanaal!')

video_id = video_entry['id']
title = video_entry.get('title', 'Unknown')
duration_sec = video_entry.get('duration', 0)
print(f'Gevonden: {title}')
print(f'Duur: {duration_sec // 60} minuten')
print(f'Downloaden...')

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
dl_opts = {
    'format': 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best',
    'outtmpl': f'{TEMP_DIR}/ranst_{ts}.%(ext)s',
    'quiet': False, 'merge_output_format': 'mp4',
}
with yt_dlp.YoutubeDL(dl_opts) as ydl2:
    ydl2.download([f'https://www.youtube.com/watch?v={video_id}'])

video_path = None
for f in os.listdir(TEMP_DIR):
    if f.startswith(f'ranst_{ts}') and f.endswith(('.mp4', '.mkv', '.webm')):
        video_path = f'{TEMP_DIR}/{f}'
        break

if not video_path:
    raise FileNotFoundError('Download mislukt!')

print(f'Video gedownload: {os.path.basename(video_path)}')
print(f'Size: {os.path.getsize(video_path) / (1024*1024):.1f} MB')

## 5. STREAMING PROCESSOR

In [ ]:
class StreamingProcessor:
    def __init__(self, video_path):
        self.video_path = video_path
        self.timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        self.session_id = f'ranst_news_{self.timestamp}'
        self.all_segments = []
        self.news_alerts = []
        self.output_lock = threading.Lock()
        self._start_time = None
        self._last_update_time = None

    def split_into_chunks(self, chunk_minutes=5):
        result = subprocess.run([
            'ffprobe', '-v', 'error',
            '-show_entries', 'format=duration',
            '-of', 'default=noprint_wrappers=1:nokey=1',
            self.video_path
        ], capture_output=True, text=True)
        duration = float(result.stdout.strip())
        chunk_size = chunk_minutes * 60
        chunks = []
        for i in range(0, int(duration), int(chunk_size)):
            chunks.append({
                'num': len(chunks) + 1,
                'start': i,
                'end': min(i + chunk_size, duration),
                'duration': min(chunk_size, duration - i)
            })
        return chunks

    def extract_chunk(self, chunk_info):
        num = chunk_info['num']
        out_file = f'{TEMP_DIR}/chunk_{num:03d}.wav'
        subprocess.run([
            'ffmpeg', '-y', '-i', self.video_path,
            '-ss', str(chunk_info['start']),
            '-t', str(chunk_info['duration']),
            '-ac', '1', '-ar', '16000', '-q:a', '9', out_file
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        return out_file if os.path.exists(out_file) else None

    def process_chunk(self, chunk_info, audio_file):
        try:
            segments = transcribe_chunk(audio_file)
            if not segments:
                return {'alert': None, 'segments': []}
            turns = diarize_chunk(audio_file)
            enriched = []
            for seg in segments:
                enriched.append({
                    'chunk': chunk_info['num'],
                    'speaker': match_speaker_to_segment(seg, turns),
                    'text': seg['text'].strip(),
                    'start': chunk_info['start'] + seg['start'],
                    'end': chunk_info['start'] + seg['end']
                })
            chunk_text = ' '.join([s['text'] for s in enriched])
            news = detect_news(chunk_text)
            alert = None
            if news['is_newsworthy']:
                alert = {
                    'timestamp': datetime.now().isoformat(),
                    'chunk': chunk_info['num'],
                    'time': f"{chunk_info['start']/60:.0f}m-{chunk_info['end']/60:.0f}m",
                    'score': news['score'],
                    'category': news['category'],
                    'reason': news['reason'],
                    'speakers': list(set([s['speaker'] for s in enriched])),
                    'text_preview': chunk_text[:300]
                }
            return {'alert': alert, 'segments': enriched}
        except Exception as e:
            print(f'  Chunk {chunk_info["num"]} error: {e}')
            return {'alert': None, 'segments': []}

    def _print_progress(self, chunks_done, total_chunks):
        now = _time.time()
        elapsed = now - self._start_time
        pct = (chunks_done / total_chunks) * 100
        if chunks_done > 0:
            per_chunk = elapsed / chunks_done
            remaining = per_chunk * (total_chunks - chunks_done)
            eta_min = remaining / 60
            elapsed_min = elapsed / 60
            print(f'\n{"="*60}')
            print(f'VOORTGANG: {pct:.0f}% ({chunks_done}/{total_chunks} chunks)')
            print(f'Verstreken: {elapsed_min:.1f} min | Resterend: {eta_min:.1f} min')
            print(f'Nieuws alerts: {len(self.news_alerts)}')
            print(f'{"="*60}\n')
        else:
            print(f'\nVOORTGANG: 0% - Bezig met starten...\n')
        self._last_update_time = now

    def process_streaming(self):
        print(f'\n{"="*60}')
        print('STREAMING PIPELINE - GPU OPTIMIZED')
        print(f'{"="*60}')
        self._start_time = _time.time()
        self._last_update_time = self._start_time
        chunks = self.split_into_chunks()
        total_chunks = len(chunks)
        print(f'\nVideo: {os.path.basename(self.video_path)}')
        print(f'Chunks: {total_chunks} x 5min')
        print(f'Device: {DEVICE.upper()}')
        print(f'Updates elke 5 minuten\n')
        print('Stap 1/2: Audio extractie...')
        chunk_files = {}
        with ThreadPoolExecutor(max_workers=2) as executor:
            futures = {executor.submit(self.extract_chunk, c): c for c in chunks}
            for future in tqdm(as_completed(futures), total=total_chunks, desc='Extracting'):
                chunk = futures[future]
                chunk_files[chunk['num']] = future.result()
        print('\nStap 2/2: Transcriptie + analyse...')
        chunks_done = 0
        self._print_progress(0, total_chunks)
        for chunk in chunks:
            if chunk['num'] not in chunk_files or not chunk_files[chunk['num']]:
                chunks_done += 1
                continue
            result = self.process_chunk(chunk, chunk_files[chunk['num']])
            with self.output_lock:
                self.all_segments.extend(result['segments'])
                if result['alert']:
                    self.news_alerts.append(result['alert'])
                    print(f'  ALERT chunk {chunk["num"]}: {result["alert"]["reason"]}')
            try:
                os.remove(chunk_files[chunk['num']])
            except:
                pass
            chunks_done += 1
            now = _time.time()
            if now - self._last_update_time >= 300 or chunks_done == total_chunks:
                self._print_progress(chunks_done, total_chunks)
        self.save_results()

    def save_results(self):
        elapsed_total = (_time.time() - self._start_time) / 60
        print(f'\n{"="*60}')
        print(f'VERWERKING COMPLEET in {elapsed_total:.1f} minuten')
        print(f'{"="*60}')
        output_file = f'{OUTPUT_DIR}/{self.session_id}.json'
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump({
                'metadata': {
                    'source': os.path.basename(self.video_path),
                    'processed_at': datetime.now().isoformat(),
                    'processing_time_minutes': round(elapsed_total, 1),
                    'total_segments': len(self.all_segments),
                    'total_alerts': len(self.news_alerts),
                    'session_id': self.session_id
                },
                'segments': self.all_segments,
                'news_alerts': self.news_alerts
            }, f, ensure_ascii=False, indent=2)
        transcript_file = f'{OUTPUT_DIR}/{self.session_id}_transcript.txt'
        with open(transcript_file, 'w', encoding='utf-8') as f:
            f.write('RANST RAADSVERGADERING - TRANSCRIPT\n')
            f.write(f'Bron: {os.path.basename(self.video_path)}\n')
            f.write(f'Verwerkt: {datetime.now().strftime("%Y-%m-%d %H:%M")}\n')
            f.write('=' * 60 + '\n\n')
            current_speaker = None
            for seg in self.all_segments:
                if seg['speaker'] != current_speaker:
                    current_speaker = seg['speaker']
                    mins = int(seg['start'] // 60)
                    secs = int(seg['start'] % 60)
                    f.write(f'\n[{mins:02d}:{secs:02d}] {current_speaker}:\n')
                f.write(f'  {seg["text"]}\n')
        for i, alert in enumerate(self.news_alerts, 1):
            email_file = f'{OUTPUT_DIR}/{self.session_id}_email_alert_{i:03d}.txt'
            with open(email_file, 'w', encoding='utf-8') as f:
                f.write('RANST RAADSVERGADERING - NIEUWSALERT\n')
                f.write('=' * 60 + '\n')
                f.write(f'TIME: {alert["time"]}\n')
                f.write(f'SCORE: {alert["score"]:.2f}\n')
                f.write(f'CATEGORY: {alert["category"]}\n\n')
                f.write(f'REASON: {alert["reason"]}\n\n')
                speakers_str = ', '.join(alert['speakers'])
                f.write(f'SPEAKERS: {speakers_str}\n\n')
                f.write(f'TEXT PREVIEW:\n{alert["text_preview"]}\n')
        print(f'JSON: {output_file}')
        print(f'Transcript: {transcript_file}')
        print(f'Segmenten: {len(self.all_segments)}')
        print(f'Nieuws alerts: {len(self.news_alerts)}')
        return output_file

print('StreamingProcessor ready')

## 6. RUN

In [ ]:
print(f'Video: {video_path}')
print(f'Output: {OUTPUT_DIR}\n')

processor = StreamingProcessor(video_path)
processor.process_streaming()

## 7. DOWNLOAD RESULTATEN

In [ ]:
import shutil
from google.colab import files

output_files = sorted([f for f in os.listdir(OUTPUT_DIR) if f.startswith('ranst_news_')], reverse=True)
print(f'{len(output_files)} output bestanden:\n')
for file in output_files:
    filepath = f'{OUTPUT_DIR}/{file}'
    print(f'  {file} ({os.path.getsize(filepath) / 1024:.1f}KB)')

shutil.make_archive('/content/ranst_output', 'zip', OUTPUT_DIR)
files.download('/content/ranst_output.zip')
print('Download gestart!')

## 8. PREVIEW RESULTATEN

In [ ]:
latest_json = f'{OUTPUT_DIR}/{processor.session_id}.json'
with open(latest_json) as f:
    results = json.load(f)

print(f'\n{"="*60}')
print('RESULTATEN SAMENVATTING')
print(f'{"="*60}')
print(f'Segmenten: {results["metadata"]["total_segments"]}')
print(f'Nieuws alerts: {results["metadata"]["total_alerts"]}')
print(f'Verwerkingstijd: {results["metadata"]["processing_time_minutes"]} min')

if results['news_alerts']:
    print('\nTOP ALERTS:')
    for i, alert in enumerate(results['news_alerts'][:5], 1):
        print(f'\n  {i}. [{alert["time"]}] Score: {alert["score"]:.2f}')
        print(f'     {alert["reason"]}')
        speakers_str = ', '.join(alert['speakers'])
        print(f'     Speakers: {speakers_str}')
        print(f'     Preview: {alert["text_preview"][:100]}...')

## 9. SHUTDOWN (GPU vrijgeven)

In [ ]:
# Voer ALLEEN uit als je klaar bent met downloaden!
import time
print('Runtime wordt over 10 seconden afgesloten...')
time.sleep(10)
os.system('kill -9 -1')